In [ ]:
# ============================================================
# PHASE 1, 2, 3
# Credit Card Product Intelligence Dataset
# Data Validation + EDA + Clustering
# ============================================================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv("synthetic_canadian_consumer_credit_card_product_intelligence_v3.csv")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

In [ ]:
# ============================================================
# PHASE 1 — DATA VALIDATION
# ============================================================

# Check missing values
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

print("Missing Values:")
print(missing_values)

if missing_values.empty:
    print("No missing values found.")

In [ ]:
# Check duplicate rows
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

In [ ]:
# Check data types
print("Data Types:")
print(df.dtypes)

In [ ]:
# Remove primary and secondary product type columns
df.drop(
    columns=["primary_product_type", "secondary_product_type"],
    inplace=True
)

print("Columns removed successfully.")

In [ ]:
# Check category distributions
categorical_columns = [
    "issuer_type",
    "card_category",
    "network_provider",
    "active_status",
    "reward_program_type",
    "reward_currency_type",

]

for col in categorical_columns:
    print("\n", col)
    print(df[col].value_counts())

In [ ]:
# Business logic validation

print("Average annual fee by card category:")
print(df.groupby("card_category")["annual_fee_cad"].mean().sort_values(ascending=False))

print("\nAverage credit score by card category:")
print(df.groupby("card_category")["minimum_credit_score"].mean().sort_values(ascending=False))

print("\nAverage travel score by card category:")
print(df.groupby("card_category")["travel_feature_level"].mean().sort_values(ascending=False))

print("\nAverage digital score by card category:")
print(df.groupby("card_category")["digital_feature_level"].mean().sort_values(ascending=False))

In [ ]:
# Save validation summary
validation_summary = pd.DataFrame({
    "Total Rows": [df.shape[0]],
    "Total Columns": [df.shape[1]],
    "Missing Values": [df.isnull().sum().sum()],
    "Duplicate Rows": [duplicates]
})

validation_summary.to_csv("phase1_data_validation_summary.csv", index=False)

print("Phase 1 validation summary saved.")

In [ ]:
# ============================================================
# PHASE 2 — EXPLORATORY DATA ANALYSIS
# ============================================================

# Card category distribution
plt.figure(figsize=(12,6))
df["card_category"].value_counts().plot(kind="bar")
plt.title("Card Category Distribution")
plt.xlabel("Card Category")
plt.ylabel("Number of Products")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Issuer type distribution
plt.figure(figsize=(8,5))
df["issuer_type"].value_counts().plot(kind="bar")
plt.title("Issuer Type Distribution")
plt.xlabel("Issuer Type")
plt.ylabel("Number of Products")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Annual fee distribution
plt.figure(figsize=(8,5))
plt.hist(df["annual_fee_cad"], bins=20)
plt.title("Annual Fee Distribution")
plt.xlabel("Annual Fee CAD")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()

In [ ]:
# Average annual fee by category
avg_fee = df.groupby("card_category")["annual_fee_cad"].mean().sort_values(ascending=False)

plt.figure(figsize=(12,6))
avg_fee.plot(kind="bar")
plt.title("Average Annual Fee by Card Category")
plt.xlabel("Card Category")
plt.ylabel("Average Annual Fee CAD")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Reward program distribution
plt.figure(figsize=(8,5))
df["reward_program_type"].value_counts().plot(kind="bar")
plt.title("Reward Program Type Distribution")
plt.xlabel("Reward Program Type")
plt.ylabel("Number of Products")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Product intelligence score averages
score_columns = [
 
    "cashback_feature_level",
    "lifestyle_feature_level",
    "travel_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level",
    "security_protection_feature_level",
    "premium_benefits_feature_level"
]

score_means = df[score_columns].mean().sort_values(ascending=False)

plt.figure(figsize=(10,5))
score_means.plot(kind="bar")
plt.title("Average Product Intelligence Scores")
plt.xlabel("Score Type")
plt.ylabel("Average Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric columns
# Select key features only

selected_features = [
    "annual_fee_cad",
    "minimum_income_requirement",
    "minimum_credit_score",
    "purchase_interest_rate",
    "base_reward_rate",
    "welcome_bonus_value_cad",
    "cashback_feature_level",
    "lifestyle_feature_level",
    "travel_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level",
    "security_protection_feature_level",
    "premium_benefits_feature_level"
]

corr = df[selected_features].corr()

plt.figure(figsize=(10,8))
plt.imshow(corr)
plt.colorbar()
plt.title("Key Product Features Correlation Matrix")

plt.xticks(
    range(len(selected_features)),
    selected_features,
    rotation=45,
    ha='right'
)

plt.yticks(
    range(len(selected_features)),
    selected_features
)

plt.tight_layout()
plt.show()

In [ ]:
# Save final dataset with cluster labels

df.to_csv("credit_card_dataset_with_clusters_v2.csv", index=False)

print("Final dataset with clusters saved as:")
print("credit_card_dataset_with_clusters_v2.csv")

In [ ]:
# ============================================================
# PHASE 3 — PRODUCT SEGMENTATION / CLUSTERING
# ============================================================

# Select features for clustering
clustering_features = [
    "annual_fee_cad",
    "minimum_income_requirement",
    "minimum_credit_score",
    "foreign_transaction_fee_percent",
    "purchase_interest_rate",
    "base_reward_rate",
    "cashback_feature_level",
    "lifestyle_feature_level",
    "travel_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level",
    "security_protection_feature_level",
    "premium_benefits_feature_level"
]

X = df[clustering_features]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features prepared for clustering.")

In [ ]:
# Find best number of clusters using Elbow Method

inertia = []

K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette scores

silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8,5))
plt.plot(K_range, silhouette_scores, marker="o")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

for k, score in zip(K_range, silhouette_scores):
    print("k =", k, "Silhouette Score =", round(score, 3))

In [ ]:
# Train final clustering model
# You can change n_clusters after viewing elbow/silhouette results

best_k = 6

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["product_cluster"] = kmeans.fit_predict(X_scaled)

print("Clustering completed.")
print(df["product_cluster"].value_counts().sort_index())

In [ ]:
# Analyze clusters

cluster_profile = df.groupby("product_cluster")[clustering_features].mean()

print("Cluster Profiles:")
display(cluster_profile)

cluster_profile.to_csv("phase2_cluster_profiles.csv")

In [ ]:
# View dominant product categories in each cluster

cluster_category = pd.crosstab(df["product_cluster"], df["card_category"])

print("Card category distribution by cluster:")
display(cluster_category)

cluster_category.to_csv("phase2_cluster_category_distribution.csv")

In [ ]:
# PCA visualization of clusters

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df["pca_1"] = X_pca[:, 0]
df["pca_2"] = X_pca[:, 1]

plt.figure(figsize=(10,7))
plt.scatter(df["pca_1"], df["pca_2"], c=df["product_cluster"], alpha=0.7)
plt.title("Product Clusters Visualized with PCA")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(label="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
!pip install umap-learn

In [ ]:
import umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

plt.figure(figsize=(10,7))

plt.scatter(
    X_umap[:,0],
    X_umap[:,1],
    c=df["product_cluster"],
    alpha=0.7
)

plt.title("UMAP Visualization of Product Clusters")
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
cluster_profile.round(2)

In [ ]:
cluster_profile.T

In [ ]:
# Define business names for each cluster
cluster_names = {
    0: "Digital-First Cards",
    1: "Rewards & Travel Cards",
    2: "Sustainable Rewards",
    3: "Premium Travel Elite",
    4: "Student & Financial Wellness",
    5: "Everyday Cashback & Lifestyle"
}

# Create a new column with business-friendly names
df['cluster_name'] = df['product_cluster'].map(cluster_names)

# Check the results
df[['product_cluster', 'cluster_name']].head()

# Save final dataset with cluster information

#df.to_csv(
#    "credit_card_product_intelligence_with_clusters.csv",
#    index=False
#)

#print("Dataset saved successfully!")
#print("File name: credit_card_product_intelligence_with_clusters.csv")

In [ ]:
#Verify the distribution
df.groupby('cluster_name').size().sort_values(ascending=False)

In [ ]:
#Save Database
output_file = "synthetic_credit_card_products_clustered_v2.csv"

df.to_csv(output_file, index=False)

print(f"Dataset saved successfully as: {output_file}")

In [ ]:

import joblib

# Business interpretation for each numeric cluster
cluster_names = {
    0: "Digital-First Cards",
    1: "Rewards & Travel Cards",
    2: "Sustainable Rewards",
    3: "Premium Travel Elite",
    4: "Student & Financial Wellness",
    5: "Everyday Cashback & Lifestyle"
}

# Save fitted clustering components
joblib.dump(kmeans, "credit_card_clustering_model.pkl")
joblib.dump(scaler, "clustering_scaler.pkl")
joblib.dump(list(clustering_features), "clustering_feature_columns.pkl")
joblib.dump(cluster_names, "cluster_name_mapping.pkl")

print("Clustering model files saved successfully.")

In [ ]:
# Load files to verify that they were saved correctly
loaded_kmeans = joblib.load("credit_card_clustering_model.pkl")
loaded_scaler = joblib.load("clustering_scaler.pkl")
loaded_features = joblib.load("clustering_feature_columns.pkl")
loaded_cluster_names = joblib.load("cluster_name_mapping.pkl")

print("Number of clusters:", loaded_kmeans.n_clusters)
print("Number of clustering features:", len(loaded_features))
print("Cluster names:", loaded_cluster_names)

clustering_interpretation = """
# Product Clustering Interpretation

## Overview

K-Means clustering was applied to segment the synthetic Canadian credit card
product portfolio into six distinct groups. The clustering analysis used product
pricing, eligibility requirements, reward structures, digital capabilities,
subscription benefits, sustainability features, financial wellness offerings,
security protections, travel features, and premium benefits.

The Elbow Method and Silhouette Analysis were used to evaluate different values
of `k`. Although the highest silhouette score was observed at `k = 2`, that
solution produced broad segments that were not sufficiently detailed for product
portfolio analysis.

A six-cluster solution was selected because it provided the best balance among:

- Statistical cluster quality
- Business interpretability
- Portfolio coverage
- Meaningful product differentiation
- Alignment with realistic Canadian credit card strategies

The resulting clusters represent product strategies commonly found across
Canadian banks, credit unions, fintech companies, digital-first issuers, and
co-branded card programs.

---

## Cluster 0: Digital-First Cards

**Portfolio size:** 297 products

### Key Characteristics

- Average annual fee: **$32.64**
- Minimum income requirement: **$17,010**
- Minimum credit score: **674**
- Highest digital feature level: **70.8**
- Highest subscription feature level: **35.5**
- Financial wellness feature level: **45.7**
- Low travel feature level: **8.2**

### Product Profile

This cluster contains affordable digital-first credit card products with strong
mobile and subscription-related capabilities. Common features may include virtual
cards, instant issuance, mobile wallet integration, real-time notifications,
subscription management, budgeting tools, and spending insights.

These products are positioned for digitally engaged customers seeking convenient,
low-cost, and technology-enabled credit card experiences.

---

## Cluster 1: Rewards & Travel Cards

**Portfolio size:** 317 products

### Key Characteristics

- Average annual fee: **$69.05**
- Minimum income requirement: **$38,927**
- Minimum credit score: **706**
- Average base reward rate: **1.71%**
- Travel feature level: **28.4**
- Digital feature level: **44.6**
- Premium benefits level: **7.3**
- Foreign transaction fee: **1.78%**

### Product Profile

This cluster represents mainstream rewards products that combine general reward
earning with moderate travel benefits. These cards typically serve established
customers seeking flexible rewards without the high fees and eligibility
requirements associated with premium travel products.

The segment occupies a middle-market position between everyday rewards cards and
premium travel cards.

---

## Cluster 2: Sustainable Rewards

**Portfolio size:** 69 products

### Key Characteristics

- Average annual fee: **$33.20**
- Minimum income requirement: **$22,464**
- Minimum credit score: **696**
- Highest sustainability feature level: **83.3**
- Digital feature level: **48.9**
- Cashback feature level: **30.7**
- Low premium benefits level: **0.9**

### Product Profile

This cluster contains sustainability-focused products that emphasize
environmentally responsible spending and green reward programs.

Potential features include carbon-footprint tracking, sustainable merchant
rewards, environmental donations, recycled card materials, and incentives for
low-carbon purchasing behaviour.

The segment targets environmentally conscious customers while maintaining
relatively accessible pricing and eligibility requirements.

---

## Cluster 3: Premium Travel Elite

**Portfolio size:** 116 products

### Key Characteristics

- Average annual fee: **$531.76**
- Minimum income requirement: **$90,776**
- Minimum credit score: **769**
- Highest travel feature level: **86.7**
- Highest premium benefits level: **70.5**
- Highest base reward rate: **2.32%**
- Lowest foreign transaction fee: **0.75%**
- Lifestyle feature level: **27.3**

### Product Profile

This cluster represents premium travel credit cards designed for affluent
customers and frequent travellers.

Typical benefits may include airport lounge access, comprehensive travel
insurance, concierge services, hotel and airline rewards, travel credits,
priority services, baggage protection, and reduced foreign transaction fees.

These products are positioned at the high end of the market and are designed to
support premium customer acquisition, engagement, and retention.

---

## Cluster 4: Student & Financial Wellness

**Portfolio size:** 213 products

### Key Characteristics

- Average annual fee: **$25.71**
- Minimum income requirement: **$15,469**
- Minimum credit score: **673**
- Lowest purchase interest rate: **13.96%**
- Highest financial wellness feature level: **54.6**
- Low cashback feature level: **16.1**
- Low travel feature level: **7.3**
- Low premium benefits level: **0.4**

### Product Profile

This cluster represents accessible credit card products focused on affordability,
credit building, and financial wellness.

Common features may include budgeting tools, spending alerts, credit-score
monitoring, payment reminders, financial education, and lower eligibility
requirements.

These products are suitable for students, young adults, first-time cardholders,
and customers seeking tools to support responsible credit use.

---

## Cluster 5: Everyday Cashback & Lifestyle

**Portfolio size:** 345 products

### Key Characteristics

- Average annual fee: **$55.48**
- Minimum income requirement: **$36,130**
- Minimum credit score: **684**
- Highest cashback feature level: **58.2**
- Highest lifestyle feature level: **36.0**
- Average base reward rate: **1.79%**
- Digital feature level: **51.1**
- Financial wellness feature level: **25.7**

### Product Profile

This cluster contains mass-market credit cards designed to reward routine
consumer spending.

Rewards are commonly concentrated in categories such as groceries, dining,
entertainment, transportation, shopping, and other everyday purchases.

These products are positioned for customers seeking practical cashback value and
lifestyle benefits without the cost or eligibility requirements of premium
travel cards.

---

## Business Insights

The clustering analysis identified six meaningful product portfolio segments:

1. Digital-First Cards
2. Rewards & Travel Cards
3. Sustainable Rewards
4. Premium Travel Elite
5. Student & Financial Wellness
6. Everyday Cashback & Lifestyle

The results indicate that pricing, eligibility requirements, rewards,
sustainability, digital capabilities, financial wellness, and premium travel
benefits create distinct product strategies within the synthetic Canadian credit
card market.

Security protection levels were relatively consistent across the clusters. This
suggests that security features operate primarily as standard portfolio
requirements rather than as major product differentiators.

The six clusters can support:

- Product portfolio intelligence
- Market gap analysis
- Product similarity analysis
- Product identity classification
- Credit card recommendation systems
- New product concept evaluation
- Future product innovation initiatives

The clustering output will also provide the target product identities used in the
subsequent classification stage of the project.
"""

# Save as a Markdown file for GitHub
with open(
    "PRODUCT_CLUSTERING_INTERPRETATION.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(clustering_interpretation.strip())

print(
    "Product clustering interpretation saved as "
    "PRODUCT_CLUSTERING_INTERPRETATION.md"
)